Import Library

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import numpy as np
import cv2
from scipy.fftpack import dct
import matplotlib.pyplot as plt
from PIL import Image
import os
import kagglehub
from google.colab import files

# Cek ketersediaan GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device aktif: {device}")

✅ Device aktif: cpu


Download Dataset

In [3]:
print("⬇️ Sedang mendownload dataset...")
dataset_path = kagglehub.dataset_download("aryankaushik005/custom-dataset")

print(f"✅ Path dasar download: {dataset_path}")

# Mencari folder 'content'
target_dir = os.path.join(dataset_path, "content")

# Verifikasi apakah folder content benar ada
if os.path.exists(target_dir):
    print(f"📂 FOLDER DATASET VALID DITEMUKAN: {target_dir}")
    # Cek isi sekilas
    print(f"   Isi: {os.listdir(target_dir)}")
else:
    print("⚠️ Folder 'content' tidak ditemukan, mencoba path root...")
    target_dir = dataset_path

# Pastikan PyTorch bisa membaca folder ini
try:
    # Test dummy untuk cek kelas
    dummy_data = datasets.ImageFolder(target_dir)
    print(f"✅ KELAS TERDETEKSI: {dummy_data.classes}")
    print("   (PyTorch otomatis menggabungkan semua folder dalam 'fake_images' menjadi satu kelas)")
except Exception as e:
    print(f"❌ Error Struktur Folder: {e}")

⬇️ Sedang mendownload dataset...
Download already complete (5113728025 bytes).
Extracting files...
✅ Path dasar download: /root/.cache/kagglehub/datasets/aryankaushik005/custom-dataset/versions/1
📂 FOLDER DATASET VALID DITEMUKAN: /root/.cache/kagglehub/datasets/aryankaushik005/custom-dataset/versions/1/content
   Isi: ['fake_images', 'progress.json', 'real_images']
✅ KELAS TERDETEKSI: ['fake_images', 'real_images']
   (PyTorch otomatis menggabungkan semua folder dalam 'fake_images' menjadi satu kelas)


Definisi Kelas "Novelty" (Analisis Krominansi)

In [4]:
class ChrominanceFrequencyTransform:
    """
    NOVELTY: Mengubah gambar ke domain frekuensi berbasis warna (YCbCr).
    Tujuannya: Mendeteksi jejak upsampling GAN pada channel warna.
    """
    def __call__(self, img):
        img_np = np.array(img)

        # Handle gambar grayscale
        if len(img_np.shape) < 3:
            img_np = cv2.cvtColor(img_np, cv2.COLOR_GRAY2RGB)

        # 1. Konversi RGB ke YCrCb
        img_ycc = cv2.cvtColor(img_np, cv2.COLOR_RGB2YCrCb)

        # 2. Ambil channel Cr & Cb (Warna Merah & Biru)
        chroma_1 = img_ycc[:,:,1]
        chroma_2 = img_ycc[:,:,2]

        # 3. Transformasi DCT (Matematika Frekuensi)
        dct_1 = dct(dct(chroma_1.T, norm='ortho').T, norm='ortho')
        dct_2 = dct(dct(chroma_2.T, norm='ortho').T, norm='ortho')

        # 4. Log Transform (Agar terlihat polanya)
        dct_combined = (np.log(np.abs(dct_1) + 1e-12) + np.log(np.abs(dct_2) + 1e-12)) / 2
        dct_norm = (dct_combined - np.min(dct_combined)) / (np.max(dct_combined) - np.min(dct_combined))

        # 5. Stack jadi 3 channel untuk input ResNet
        return np.stack([dct_norm] * 3, axis=2).astype(np.float32)

Setup Data Loader & Model

In [5]:
# Pipeline Data
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    ChrominanceFrequencyTransform(), # <--- Panggil Inovasi
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load Dataset ke Memory
train_data = datasets.ImageFolder(target_dir, transform=data_transforms)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

# Catat Label Mapping (PENTING UNTUK TESTING)
class_names = train_data.classes
print(f"📊 Mapping Label: {train_data.class_to_idx}")
# Biasanya: {'fake_images': 0, 'real_images': 1}

# Setup Model ResNet18
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2) # Ubah output jadi 2 (Fake vs Real)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

📊 Mapping Label: {'fake_images': 0, 'real_images': 1}


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 122MB/s]


Training

In [ ]:
num_epochs = 10
print(f"\n MULAI TRAINING... ({num_epochs} Epochs)")

model.train()
for epoch in range(num_epochs):
    running_loss = 0.0
    correct = 0
    total = 0

    print(f"Sedang memproses Epoch {epoch+1}...", end="")

    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        if i % 50 == 0:
            print(".", end="") # Indikator loading

    epoch_acc = 100 * correct / total
    print(f"\n>> SELESAI Epoch {epoch+1} | Loss: {running_loss/len(train_loader):.4f} | Akurasi: {epoch_acc:.2f}%")

print("\n✅ Training Selesai!")


 MULAI TRAINING... (10 Epochs)
Sedang memproses Epoch 1....

Simpan & Download Model

In [ ]:
save_name = "model_deepfake_chroma.pth"
torch.save(model.state_dict(), save_name)
print(f"💾 Model disimpan di Colab sebagai: {save_name}")

print("⬇️ Mendownload file otomatis...")
try:
    files.download(save_name)
except:
    print("⚠️ Download otomatis gagal. Silakan download manual")